# Multiple Dispatch
This library makes liberal use of Plum, an implementation of multiple dispatch for Python. 
Multiple dispatch allows you to implement multiple methods for the same function, where the methods specify the types of their arguments.
When a implementation for a given argument type is not found, the dispatch system will search for an implementation for a parent type, and so on, until it reaches the root type.

I like multiple dispatch for a few reasons.
It enables the sort of hierarchical polymorphism that is common in object-oriented languages, but without the need for coupling class definitions to function definitions.
This helps keep code modular and reusable.
Since an error is raised when no implementation for a given argument type is found, it also effectively provides runtime type checking, which is useful for debugging and documentation.
Another key feature is the extensibility of the dispatch system.
With multiple dispatch, adding new functionality generally involves adding new functions, rather than modifying existing classes. This makes it easier to extend capabilities without touching old code, making it more maintainable and reducing the risk of breaking changes.
This is incredibly helpful for managing a library like this one, where the number of possible combinations of arguments is large and growing.

This notebook will document (and test) the multiple dispatch system as it used by this library, documenting design practices and clarifying intended behavior.

In [1]:
from plum import dispatch, NotFoundLookupError

@dispatch
def f(x: str):
    return "This is a string!"
    

@dispatch
def f(x: int):
    return "This is an integer!"

In [2]:
f("1")

'This is a string!'

In [3]:
f(1)

'This is an integer!'

In [4]:
import pytest

with pytest.raises(NotFoundLookupError):
    f(1.0)

## Avoiding Overly Narrow Dispatch
Multiple dispatch shouldn't necessarily be used everywhere.
The narrow argument types anti-pattern refers to the situation in which the types of the arguments are too narrowly specified, causing the function to be less useful unnecessarily.
A function that would work just as well with a parent type is unnecessarily limited through narrow dispatch.
To avoid this issue, consider the following options:

### Start Generic, Refine Later
You start by implementing a generic function and later add more specific methods as the requirements become clear. This allows you to start with a flexible approach and refine it based on actual needs.

In [5]:
@dispatch
def f(x):
    return "This is something!"

In [6]:
f(1.0)

'This is something!'

We can also make some arguments generic while others are more specific.

In [7]:
@dispatch
def f(x, y: int):
    return x+y

f(1.0, 2)

3.0

### Use Type Hierarchies
The above has the downside in that it doesn't do any type checking. 
Type checking at critical points can catch inconsistencies before they propagate too far into the system.
A way to avoid this without overly narrowing the dispatch is to use type hierarchies.
Specify that a function works with any type that inherits from a specific superclass or implements a particular interface. This gives you some level of type checking while still allowing for extensibility.

In [9]:
from numbers import Number

@dispatch
def f(x: Number, y: Number):
    return 'Multiply two numbers.'


@dispatch
def f(x: float, y: float):
    return 'Multiply two floats.'


@dispatch
def f(x: int, y: int):
    return 'Multiply two ints.'

In [10]:
f(1, 2.)

'Multiply two numbers.'